In [68]:
from google.colab import userdata
import os

openrouter_key = userdata.get("OPENROUTER_API_KEY")
tavily_key = userdata.get("TAVILY_API_KEY")

assert openrouter_key, "OPENROUTER_API_KEY not found in Colab Secrets"
assert tavily_key, "TAVILY_API_KEY not found in Colab Secrets"

os.environ["OPENROUTER_API_KEY"] = openrouter_key.strip()
os.environ["TAVILY_API_KEY"] = tavily_key.strip()

print("OpenRouter loaded:", os.environ["OPENROUTER_API_KEY"][:12])
print("Tavily loaded:", os.environ["TAVILY_API_KEY"][:12])

OpenRouter loaded: sk-or-v1-427
Tavily loaded: tvly-dev-3JD


In [51]:
!pip install -U langchain langchain_openai langchain_community requests python-dotenv beautifulsoup4

In [52]:
from google.colab import userdata
import os

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [53]:
import os
import requests
from bs4 import BeautifulSoup

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from langchain_community.tools.tavily_search import TavilySearchResults

In [54]:
internet_search = TavilySearchResults()

In [55]:
!pip install -U langchain-tavily

In [56]:
from langchain_tavily import TavilySearch

internet_search = TavilySearch(max_results=3)

In [57]:
internet_search.invoke({"query": "AI in education"})

{'query': 'AI in education',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.educationnext.org/a-i-in-education-leap-into-new-era-machine-intelligence-carries-risks-challenges-promises/',
   'title': 'AI in Education - Education Next',
   'content': 'Microsoft researchers suggest that newer systems “exhibit more general intelligence than previous AI models” and are coming “strikingly close to human-level performance.” While some observers question those conclusions, the AI systems display an increasing ability to generate coherent and contextually appropriate responses, make connections between different pieces of information, and engage in reasoning processes such as inference, deduction, and analogy. AI can also provide educators with recommendations to meet student needs and help teachers reflect, plan, and improve their practice. ***Privacy concerns.*** When students or educators interact with generative-AI tools, their conversations 

In [58]:
@tool
def fetch_url(url: str) -> str:
    """Fetch readable text content from a URL."""

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            return f"FETCH_FAILED: {url}"

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        if not text or len(text.strip()) < 200:
            return f"FETCH_FAILED: {url}"

        return text[:12000]

    except Exception:
        return f"FETCH_FAILED: {url}"

In [59]:
test_url = "https://en.wikipedia.org/wiki/Artificial_intelligence_in_education"

print(fetch_url.invoke({"url": test_url})[:1000])

Artificial intelligence in education - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Search Search Appearance Donate Create account Log in Personal tools Donate Create account Log in Contents move to sidebar hide (Top) 1 History 2 Theory Toggle Theory subsection 2.1 Three paradigms of AIEd 2.2 Socio-technical imaginaries 2.3 Post-humanist and new materialist perspectives 3 Applications Toggle Applications subsection 3.1 AI-based tutoring systems 3.2 Generative AI 3.3 Emotional AI 4 Perspectives Toggle Perspectives subsection 4.1 Commercial perspectives 4.2 Institutional perspectives 4.3 Student perspectives 5 Problems Toggle Problems subsection 5.1 Over-reliance, inaccuracy, and academic integrity 5.2 Accessibility 5.3 Bias 5.4 Data privacy and intellectual property 5.5 Invisible labour and en

In [60]:
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

In [61]:
response = model.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


In [62]:
system_prompt = """
You are a research agent.

You must follow this exact process:
1. Read the user's question carefully.
2. Use internet_search to find relevant information.
3. Select exactly the top 3 URLs from the search results.
4. Call fetch_url for each of those 3 URLs at most once.
5. Use only successfully fetched page contents in the final answer.
6. If one URL fails, skip it and continue with the remaining successful sources.
7. Do not retry the same failed URL repeatedly.
8. Do not cite any URL that failed to fetch successfully.
9. Write the final answer based on fetched page contents, not just search snippets.
10. Include only the URLs that were fetched successfully in the final answer.
"""

In [63]:
agent = create_agent(
    model=model,
    tools=[internet_search, fetch_url],
    system_prompt=system_prompt,
)

In [64]:
question = "What are the health benefits of coffee?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

print(response["messages"][-1].content)

Coffee offers a range of health benefits that are supported by recent research:

- **Rich in antioxidants** that protect cells from damage and reduce inflammation【https://www.nkchealth.org/blog/health-benefits-coffee-more-just-morning-pick-me】.  
- **Enhances cognitive function** by blocking adenosine, improving mood, memory, reaction time, and may lower the risk of neurodegenerative diseases such as Alzheimer’s and Parkinson’s【https://www.nkchealth.org/blog/health-benefits-coffee-more-just-morning-pick-me】.  
- **Supports heart health**, with moderate coffee consumption linked to a reduced risk of heart disease, stroke, and heart failure【https://www.nkchealth.org/blog/health-benefits-coffee-more-just-morning-pick-me】.  
- **May lower the risk of type 2 diabetes** by improving insulin sensitivity and beta‑cell function, with studies showing up to a 50 % lower risk among regular drinkers【https://www.healthline.com/nutrition/top-evidence-based-health-benefits-of-coffee】.  
- **Protects l

In [65]:
question = "What are the health benefits of coffee?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

print(response["messages"][-1].content)

Coffee is one of the most extensively researched beverages, and the evidence points to a range of health benefits when it is consumed in moderation (generally 3–4 cups per day, or up to 400 mg of caffeine). Key findings from the sources below include:

**Energy, cognition, and mood**  
- Caffeine blocks adenosine, increasing alertness, focus, and exercise endurance.  
- Regular coffee intake is linked to improved memory, cognition, and mood, and a lower risk of depression.

**Metabolic health**  
- Moderate coffee drinking is associated with a reduced risk of type 2 diabetes; filtered coffee, in particular, can lower diabetes risk by up to 60 percent.  
- Decaffeinated coffee also offers some protective effects, though slightly weaker.

**Liver health**  
- Coffee’s antioxidant and anti‑inflammatory compounds are tied to lower liver stiffness and a reduced risk of liver disease and liver cancer.

**Cardiovascular health**  
- Studies show lower rates of heart disease, stroke, and heart

In [66]:
print(response["messages"][-1].content)

Coffee is one of the most extensively researched beverages, and the evidence points to a range of health benefits when it is consumed in moderation (generally 3–4 cups per day, or up to 400 mg of caffeine). Key findings from the sources below include:

**Energy, cognition, and mood**  
- Caffeine blocks adenosine, increasing alertness, focus, and exercise endurance.  
- Regular coffee intake is linked to improved memory, cognition, and mood, and a lower risk of depression.

**Metabolic health**  
- Moderate coffee drinking is associated with a reduced risk of type 2 diabetes; filtered coffee, in particular, can lower diabetes risk by up to 60 percent.  
- Decaffeinated coffee also offers some protective effects, though slightly weaker.

**Liver health**  
- Coffee’s antioxidant and anti‑inflammatory compounds are tied to lower liver stiffness and a reduced risk of liver disease and liver cancer.

**Cardiovascular health**  
- Studies show lower rates of heart disease, stroke, and heart

In [67]:
question = "What are the health benefits of coffee?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

print(response["messages"][-1].content)

**Health Benefits of Coffee – Summary of Evidence**

Coffee is one of the most studied beverages, and the research consistently highlights a range of health advantages when it is consumed in moderation (about 3–4 cups per day for most adults). Below are the key benefits documented in the sources I retrieved.

| Benefit | What the research says |
|--------|------------------------|
| **Boosts energy and mental alertness** | Caffeine blocks adenosine receptors in the brain, increasing dopamine and other neurotransmitters. This reduces fatigue, improves focus, and can enhance exercise endurance and performance【https://www.healthline.com/nutrition/top-evidence-based-health-benefits-of-coffee】. |
| **May lower the risk of type 2 diabetes** | Regular coffee drinking is associated with a 6 % lower risk of developing type 2 diabetes per cup consumed daily, likely because coffee helps preserve pancreatic beta‑cell function【https://www.healthline.com/nutrition/top-evidence-based-health-benefits-

In [70]:
question = "What are the health benefits of coffee?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

print(response["messages"][-1].content)

Coffee offers a range of evidence‑based health benefits when consumed in moderation (about 3–4 cups per day). According to a comprehensive review on Healthline, regular coffee intake can boost energy levels, lower the risk of type 2 diabetes, support brain health (including a reduced risk of neurodegenerative diseases such as Alzheimer’s and Parkinson’s), aid weight management, reduce the likelihood of depression, protect the liver, support heart health, contribute to greater longevity, and enhance athletic performance [https://www.healthline.com/nutrition/top-evidence-based-health-benefits-of-coffee].

The National Coffee Association also highlights that coffee’s natural antioxidant and anti‑inflammatory compounds are linked to improved memory and cognition, better mood, reduced risk of heart disease, type 2 diabetes, and several cancers (particularly liver and endometrial cancer), and overall increased longevity [https://www.aboutcoffee.org/health/health-benefits-of-coffee/].

Both s

In [71]:
print(response["messages"][-1].content)

Coffee offers a range of evidence‑based health benefits when consumed in moderation (about 3–4 cups per day). According to a comprehensive review on Healthline, regular coffee intake can boost energy levels, lower the risk of type 2 diabetes, support brain health (including a reduced risk of neurodegenerative diseases such as Alzheimer’s and Parkinson’s), aid weight management, reduce the likelihood of depression, protect the liver, support heart health, contribute to greater longevity, and enhance athletic performance [https://www.healthline.com/nutrition/top-evidence-based-health-benefits-of-coffee].

The National Coffee Association also highlights that coffee’s natural antioxidant and anti‑inflammatory compounds are linked to improved memory and cognition, better mood, reduced risk of heart disease, type 2 diabetes, and several cancers (particularly liver and endometrial cancer), and overall increased longevity [https://www.aboutcoffee.org/health/health-benefits-of-coffee/].

Both s